# SmolLM2 Fine-Tuning Notebook’u
Bu notebook’un amacı, SmolLM2-360M-Instruct modelini Türkçe açık uçlu öğrenci cevaplarının kalite sınıflandırması görevi için fine-tune etmektir.

Bu görevde model her öğrenci cevabı için iki sınıftan birini tahmin edecektir:

0 = yüksek kaliteli değil
1 = yüksek kaliteli

Bu notebook’ta kullanılacak ana veri dosyaları şunlardır:

data/processed/strategy_b/train.csv
data/processed/strategy_b/validation.csv
data/processed/strategy_b/test.csv

Bu dosyalar daha önce hazırlanmış Strategy B veri setidir. Strategy B seçilmiştir çünkü sınıf dağılımı Strategy A’ya göre daha dengelidir.

Strategy B label anlamı:

label 0 → Score ≤ 8

label 1 → Score ≥ 9

Önemli not:
Bu notebook içinde yeniden train / validation / test split yapılmayacaktır. Bütün modellerin adil karşılaştırılabilmesi için daha önce hazırlanmış aynı split dosyaları kullanılacaktır.

Bu notebook önce SmolLM2 için hazırlanacaktır. SmolLM2 fine-tuning başarılı şekilde çalıştıktan sonra aynı yapı TinyLlama, Qwen ve Gemma için uyarlanabilir.

## 1. Ortam Kontrolü
Bu bölümde Colab’da GPU açık mı kontrol ediyoruz.

Beklenen çıktı:

CUDA available: True
GPU: Tesla T4

Eğer CUDA available False çıkarsa şu ayar yapılmalıdır:

Runtime → Change runtime type → T4 GPU

Sonra runtime yeniden başlatılıp bu hücre tekrar çalıştırılmalıdır.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU yok. Runtime ayarlarından T4 GPU açılmalı.")

## 2. Gerekli Paketlerin Kurulması
Bu bölümde fine-tuning için gerekli Python paketleri kurulacaktır.

Kullanılacak temel paketler:

- transformers: modeli ve tokenizer’ı yüklemek için
- datasets: veriyi Hugging Face Dataset formatına çevirmek için
- accelerate: GPU üzerinde eğitim sürecini yönetmek için
- peft: LoRA gibi parameter-efficient fine-tuning yöntemleri için
- bitsandbytes: düşük bellek kullanımı için
- scikit-learn: accuracy, macro F1 ve weighted F1 metriklerini hesaplamak için
- pandas: CSV dosyalarını okumak ve sonuçları kaydetmek için

Bu hücre çalıştıktan sonra runtime restart gerekirse Colab uyarı verebilir. Eğer restart gerekirse runtime yeniden başlatılıp notebook baştan çalıştırılmalıdır.

In [ ]:
!pip install transformers datasets accelerate peft bitsandbytes scikit-learn pandas -q

## 3. Kütüphaneleri İçeri Aktarma
Bu bölümde notebook boyunca kullanılacak kütüphaneler içeri aktarılır.

Burada özellikle şu işlemler için kütüphane çağırıyoruz:

- Veri okuma
- Dataset hazırlama
- Model ve tokenizer yükleme
- LoRA ayarı
- Eğitim
- Metrik hesaplama
- Sonuçları dosyaya kaydetme

In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

print("Libraries imported.")
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

## 4. Veri Dosyalarını Yükleme

Bu bölümde daha önce hazırlanmış Strategy B processed dosyaları okunacaktır.

Kullanılacak dosyalar:

data/processed/strategy_b/train.csv
data/processed/strategy_b/validation.csv
data/processed/strategy_b/test.csv

Önemli:

Bu notebook içinde yeni split yapılmayacaktır.
Aynı split dosyaları bütün modellerde kullanılacaktır.
Böylece model sonuçları adil şekilde karşılaştırılabilir.

In [ ]:
train_path = "/data/processed/strategy_b/train.csv"
validation_path = "/data/processed/strategy_b/validation.csv"
test_path = "/data/processed/strategy_b/test.csv"

print("Train exists:", os.path.exists(train_path))
print("Validation exists:", os.path.exists(validation_path))
print("Test exists:", os.path.exists(test_path))

Eğer yukarıdaki hücrede False görülürse, Colab sol taraftaki Files panelinden şu klasör yapısı oluşturulmalıdır:

data/processed/strategy_b/

Daha sonra lokal GitHub repo klasöründen şu üç dosya bu klasöre yüklenmelidir:

* train.csv
* validation.csv
* test.csv

Dosyalar yüklendikten sonra bir önceki hücre tekrar çalıştırılmalıdır.

In [ ]:
train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nTrain label distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation label distribution:")
print(validation_df["label"].value_counts().sort_index())

print("\nTest label distribution:")
print(test_df["label"].value_counts().sort_index())

## 5. Veri Yapısını Kontrol Etme

Bu bölümde verinin model eğitimi için doğru formatta olup olmadığı kontrol edilir.

Bu notebook için en önemli iki kolon:

input_text: modele verilecek metin
label: modelin tahmin edeceği sınıf

Beklenen label değerleri:

0 = yüksek kaliteli değil

1 = yüksek kaliteli

Eğer input_text veya label kolonu yoksa eğitim aşamasına geçilmemelidir.

In [ ]:
required_columns = ["input_text", "label"]

for split_name, df in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df)
]:
    print(f"\nChecking {split_name} set")

    for col in required_columns:
        print(f"{col} exists:", col in df.columns)

    print("Missing values in input_text:", df["input_text"].isna().sum())
    print("Missing values in label:", df["label"].isna().sum())
    print("Unique labels:", sorted(df["label"].unique()))

## 6. Hugging Face Dataset Formatına Çevirme

Trainer API ile eğitim yapabilmek için pandas dataframe yapısını Hugging Face Dataset formatına çeviriyoruz.

Bu aşamada sadece input_text ve label kolonları kullanılacaktır.

Diğer kolonlar rapor ve hata analizi için önemli olsa da eğitim sırasında gerekli değildir.

In [ ]:
train_dataset = Dataset.from_pandas(train_df[["input_text", "label"]])
validation_dataset = Dataset.from_pandas(validation_df[["input_text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["input_text", "label"]])

print(train_dataset)
print(validation_dataset)
print(test_dataset)

## 7. SmolLM2 Tokenizer Yükleme

Bu bölümde SmolLM2-360M-Instruct modelinin tokenizer’ı yüklenecektir.

Tokenizer, metni modelin anlayabileceği token ID’lerine dönüştürür.

Kullanılan model:

HuggingFaceTB/SmolLM2-360M-Instruct

Bu model küçük olduğu için ilk fine-tuning pipeline’ını test etmek için uygundur.

In [ ]:
model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded:", model_id)
print("Pad token:", tokenizer.pad_token)
print("Pad token id:", tokenizer.pad_token_id)

## 8. Tokenization

Bu bölümde input_text alanı tokenize edilir.

İlk deneme için max_length = 1024 kullanıyoruz.

Neden 1024?

Çünkü input metinler uzun olabilir ve GPU belleği sınırlıdır.
Daha uzun max_length değerleri daha fazla bellek kullanır.
İlk çalışan pipeline için güvenli bir değerle başlamak daha mantıklıdır.

Eğer eğitim başarıyla çalışırsa daha sonra max_length 1536 veya 2048 olarak denenebilir.

In [ ]:
MAX_LENGTH = 1024

def tokenize_function(example):
    return tokenizer(
        example["input_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_validation = validation_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_validation = tokenized_validation.rename_column("label", "labels")
tokenized_test = tokenized_test.rename_column("label", "labels")

tokenized_train.set_format("torch")
tokenized_validation.set_format("torch")
tokenized_test.set_format("torch")

print(tokenized_train)
print(tokenized_validation)
print(tokenized_test)

## 9. Modeli Sequence Classification İçin Yükleme

Bu görev bir metin üretme görevi değil, ikili sınıflandırma görevidir.

Bu nedenle modeli AutoModelForSequenceClassification ile yüklüyoruz.

num_labels = 2 olarak ayarlanır:

0 = yüksek kaliteli değil

1 = yüksek kaliteli

Bu aşamada modelin sınıflandırma head’i göreve göre yeniden öğrenilecektir.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.pad_token_id

print("Model loaded for sequence classification:", model_id)

## 10. LoRA İçin Model Modüllerini Kontrol Etme

Bu bölümde modelin içinde hangi layer/module isimlerinin bulunduğunu kontrol ediyoruz.

LoRA uygularken target_modules alanına doğru module isimlerini yazmamız gerekir.
Eğer yanlış module adı yazılırsa model eğitime başlamadan hata verir.

Bu nedenle önce modelin içindeki bazı linear layer isimlerine bakacağız.

In [ ]:
# Model içindeki module isimlerini inceleyelim
for name, module in model.named_modules():
    if "proj" in name or "linear" in name or "score" in name:
        print(name)

Bu hücrenin çıktısında genelde şuna benzer isimler görmeyi bekliyoruz:

```
model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.0.mlp.gate_proj
model.layers.0.mlp.up_proj
model.layers.0.mlp.down_proj
score
```

Burada özellikle q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj görürsek LoRA için bunları kullanabiliriz.

## 11. LoRA Ayarlarını Tanımlama

Bu bölümde modele LoRA uygulanır.

LoRA, modelin bütün ağırlıklarını güncellemek yerine bazı küçük ek parametreleri eğitir.
Bu sayede eğitim daha az GPU belleği kullanır.

Bu proje için LoRA kullanmamızın nedeni:

- Donanım sınırlı
- Modeller farklı büyüklüklerde
- Full fine-tuning daha pahalı
- Aynı yapıyı diğer modellere uyarlamak daha kolay

İlk deneme için güvenli LoRA ayarları kullanılacaktır:

r = 8
lora_alpha = 16
lora_dropout = 0.05

Bu ayarlar ilk çalışan pipeline için yeterlidir. Daha sonra gerekirse değiştirilebilir.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    bias="none"
)

model = get_peft_model(model, lora_config)

print("LoRA applied.")
model.print_trainable_parameters()

Beklenen çıktı şuna benzer olacak:

```
trainable params: ...
all params: ...
trainable%: ...
```
Burada önemli olan şu: trainable parametre oranı çok küçük olmalı. Eğer bütün model eğitiliyor gibi görünürse LoRA doğru uygulanmamış demektir.


## 12. Data Collator Tanımlama

Bu bölümde batch içindeki örneklerin aynı uzunluğa getirilmesi için data collator tanımlanır.

Metinlerin token uzunlukları farklı olduğu için padding işlemi gerekir.
DataCollatorWithPadding bunu otomatik yapar.

In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Data collator ready.")

## 13. Metrik Fonksiyonunu Tanımlama

Bu bölümde eğitim ve değerlendirme sırasında kullanılacak metrikler tanımlanır.

Raporlanacak metrikler:

- Accuracy
- Macro F1
- Weighted F1

Bu projede Macro F1 özellikle önemlidir.
Çünkü accuracy tek başına yanıltıcı olabilir.
Majority baseline zaten sadece label 0 tahmin ederek 0.6116 accuracy almıştır.
Bu nedenle modelin gerçekten iki sınıfı da öğrenip öğrenmediğini Macro F1 daha iyi gösterir.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro", zero_division=0)
    weighted_f1 = f1_score(labels, predictions, average="weighted", zero_division=0)

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

print("Metrics function ready.")

## 14. Eğitim Ayarlarını Tanımlama

Bu bölümde training ayarları tanımlanır.

İlk deneme için düşük batch size kullanıyoruz.
Bunun nedeni GPU belleğini korumaktır.

Başlangıç ayarları:

- epoch: 2
- train batch size: 2
- eval batch size: 2
- gradient accumulation: 8
- learning rate: 2e-5
- metric_for_best_model: macro_f1

Gradient accumulation sayesinde küçük batch size ile daha stabil eğitim yapılabilir.
Örneğin batch size 2 ve gradient accumulation 8 ise effective batch size yaklaşık 16 olur.

Eğer CUDA out of memory hatası alınırsa:
- per_device_train_batch_size 1 yapılabilir
- gradient_accumulation_steps artırılabilir
- max_length 1024 yerine 768 yapılabilir

In [ ]:
output_dir = "/content/smollm2_strategy_b_lora"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=2,
    learning_rate=2e-5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    fp16=True,
    report_to="none",

    save_total_limit=2
)

print("Training arguments ready.")

Not: Bazı Transformers sürümlerinde eval_strategy yerine evaluation_strategy gerekebilir. Eğer hata alırsa bu satırı şöyle değiştiririz:

In [ ]:
evaluation_strategy="epoch"

Ama yeni sürümlerde eval_strategy çalışıyor.

In [ ]:
## 15. Trainer Oluşturma

Bu bölümde Hugging Face Trainer oluşturulur.

Trainer şu parçaları bir araya getirir:

- model
- training arguments
- train dataset
- validation dataset
- tokenizer
- data collator
- metric function

Bu aşamada henüz eğitim başlamaz.
Sadece eğitim objesi hazırlanır.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer ready.")

## 16. Eğitimi Başlatma

Bu bölümde fine-tuning başlatılır.

Bu hücre zaman alabilir.
Eğitim sırasında validation sonuçları her epoch sonunda görülecektir.

Eğer CUDA out of memory hatası alınırsa:
1. Runtime restart edilir.
2. batch size 1 yapılır.
3. gradient_accumulation_steps 16 yapılır.
4. Gerekirse MAX_LENGTH 768 yapılır.
5. Notebook baştan çalıştırılır.

In [ ]:
train_result = trainer.train()

print("Training completed.")

## 17. Validation Set Üzerinde Değerlendirme

Bu bölümde eğitilmiş model validation set üzerinde değerlendirilir.

Validation sonucu, modelin eğitim sırasında nasıl performans gösterdiğini anlamak için kullanılır.
Final rapor için asıl sonuç test setinden alınacaktır.

In [ ]:
validation_results = trainer.evaluate(tokenized_validation)

print("Validation results:")
print(validation_results)

## 18. Test Set Üzerinde Final Değerlendirme

Bu bölümde model daha önce hiç görmediği test seti üzerinde değerlendirilir.

Final model performansı burada çıkan değerlerle raporlanacaktır.

Bu değerler şu sonuçlarla karşılaştırılacaktır:

- Majority class baseline
- Zero-shot SmolLM2 sonucu
- Diğer fine-tuned modeller

In [ ]:
test_results = trainer.evaluate(tokenized_test)

print("Test results:")
print(test_results)

## 19. Test Prediction Detaylarını Çıkarma

Bu bölümde test setindeki her örnek için model tahmini çıkarılır.

Bu dosya hata analizi için kullanılacaktır.
Örneğin modelin hangi cevapları yanlış sınıflandırdığı daha sonra incelenebilir.

In [ ]:
test_predictions_output = trainer.predict(tokenized_test)

test_logits = test_predictions_output.predictions
test_predictions = np.argmax(test_logits, axis=-1)

test_prediction_df = test_df.copy()
test_prediction_df["prediction"] = test_predictions
test_prediction_df["correct"] = test_prediction_df["label"] == test_prediction_df["prediction"]

print(test_prediction_df[["label", "prediction", "correct", "Score", "soru"]].head())
print("\nCorrect count:")
print(test_prediction_df["correct"].value_counts())

## 20. Sonuçları CSV Olarak Kaydetme

Bu bölümde fine-tuning sonucunda elde edilen metrikler ve prediction detayları dosyaya kaydedilir.

Kaydedilecek dosyalar:

outputs/tables/smollm2_finetune_results.csv
outputs/tables/smollm2_test_predictions.csv

Bu dosyalar daha sonra GitHub reposuna eklenmelidir.

In [ ]:
os.makedirs("/outputs/tables", exist_ok=True)

smollm2_finetune_results = pd.DataFrame([
    {
        "model": "SmolLM2-360M-Instruct",
        "training_type": "lora_finetuning",
        "dataset_strategy": "strategy_b",
        "max_length": MAX_LENGTH,
        "num_train_epochs": training_args.num_train_epochs,
        "learning_rate": training_args.learning_rate,
        "train_batch_size": training_args.per_device_train_batch_size,
        "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
        "validation_accuracy": validation_results.get("eval_accuracy"),
        "validation_macro_f1": validation_results.get("eval_macro_f1"),
        "validation_weighted_f1": validation_results.get("eval_weighted_f1"),
        "test_accuracy": test_results.get("eval_accuracy"),
        "test_macro_f1": test_results.get("eval_macro_f1"),
        "test_weighted_f1": test_results.get("eval_weighted_f1")
    }
])

smollm2_finetune_results.to_csv(
    "/outputs/tables/smollm2_finetune_results.csv",
    index=False
)

test_prediction_df.to_csv(
    "/outputs/tables/smollm2_test_predictions.csv",
    index=False
)

print("Saved:")
print("/outputs/tables/smollm2_finetune_results.csv")
print("/outputs/tables/smollm2_test_predictions.csv")

smollm2_finetune_results

## 21. Kaydedilen Dosyaları Kontrol Etme

Bu bölümde sonuç dosyalarının gerçekten oluşup oluşmadığı kontrol edilir.

Eğer True çıkarsa dosyalar Colab’dan indirilip lokal GitHub reposuna eklenebilir.

In [ ]:
print(os.path.exists("/outputs/tables/smollm2_finetune_results.csv"))
print(os.path.exists("/outputs/tables/smollm2_test_predictions.csv"))

## 22. Notebook Sonrası Yapılacaklar

Bu notebook başarıyla çalıştıktan sonra şu dosyalar lokal GitHub reposuna eklenmelidir:

* notebooks/06_finetune_smollm2.ipynb
* outputs/tables/smollm2_finetune_results.csv
* outputs/tables/smollm2_test_predictions.csv

Commit mesajı önerisi:

Add SmolLM2 LoRA fine-tuning results

Bu notebook daha sonra TinyLlama fine-tuning için template olarak kullanılabilir.
TinyLlama için değişmesi gereken ana şey model_id olacaktır.
Ancak batch size ve max_length ayarları TinyLlama için tekrar kontrol edilmelidir.